# How to use the shared code

A runnable walkthrough of the shared functions: load the data with a split, train a small model on `train`, score it on `val` with `evaluate()`, and save predictions in the shared format.

**Before running:** do the setup in the [README](../README.md) once (`pip install -e .` and `python3 scripts/prepare_data.py`). No `sys.path` lines are needed.

In [1]:
# pick up edits to src/ without restarting the kernel
%load_ext autoreload
%autoreload 2

## 1. Load the cleaned data with a split

`load_dataset(regime)` returns the cleaned articles plus a `split` column (`train` / `val` / `test`). Model on `content`; the label is `is_fake` (1 = fake).

In [2]:
from fake_news.data.load import load_dataset

df = load_dataset("random")
print(df.shape)
df.head(3)

(63018, 7)


,doc_id,title,text,content,n_words,is_fake,split
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,law enforcement on high alert following threat...,894,1,val
1,1,,Did they post their votes for Hillary already?,did they post their votes for hillary already?,8,1,test
2,2,UNBELIEVABLE! OBAMA'S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last n...",unbelievable! obama s attorney general says mo...,52,1,train


In [3]:
df.groupby("split").is_fake.agg(n="size", fake_rate="mean")

,n,fake_rate
split,,
test,9453,0.448535
train,44112,0.448449
val,9453,0.448429


## 2. Train on `train` only

Everything that learns from data (here the TF-IDF vocabulary and the classifier) is fit on `train`. This is a small toy model just to show the workflow; it is not Model A.

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from fake_news import config

train, val = df[df.split == "train"], df[df.split == "val"]

vectorizer = TfidfVectorizer(max_features=20_000, min_df=5, stop_words="english")
X_train = vectorizer.fit_transform(train.content)   # fit on train only
X_val = vectorizer.transform(val.content)           # val is only transformed

clf = LogisticRegression(max_iter=1000, random_state=config.SEED).fit(X_train, train.is_fake)
prob_fake = clf.predict_proba(X_val)[:, 1]           # P(fake) for each val article

## 3. Score on `val` with the shared `evaluate()`

Same metrics for every model: accuracy, precision, recall, F1 (fake = positive class), ROC-AUC and the confusion matrix. Tune on `val`; the `test` split is used once, at the very end, for all models together.

In [5]:
from fake_news.evaluation.metrics import evaluate

evaluate(val.is_fake, prob_fake)

{'n': 9453,
 'accuracy': 0.9222469057442082,
 'precision': 0.9099204492278896,
 'recall': 0.9174333569238028,
 'f1': 0.9136614589451427,
 'roc_auc': 0.9738517246243872,
 'tn': 4829,
 'fp': 385,
 'fn': 350,
 'tp': 3889}

## 4. Save predictions in the shared format

One file per model and regime with columns `doc_id, split, prob_fake`. Model D blends models by joining these files on `doc_id`.

In [6]:
import pandas as pd
from fake_news.evaluation.predictions import save_predictions, load_predictions, evaluate_predictions

preds = pd.DataFrame({"doc_id": val.doc_id, "split": "val", "prob_fake": prob_fake})
path = save_predictions(preds, model="example", regime="random")
print(path)

evaluate_predictions(load_predictions("example", "random"), split="val")

/home/thanos/Fake_news_recognition/data/processed/predictions/example__random.csv


{'n': 9453,
 'accuracy': 0.9222469057442082,
 'precision': 0.9099204492278896,
 'recall': 0.9174333569238028,
 'f1': 0.9136614589451427,
 'roc_auc': 0.9738517246243872,
 'tn': 4829,
 'fp': 385,
 'fn': 350,
 'tp': 3889}

## 5. Other useful pieces

**Marker effect.** `load_clean(remove_markers=False)` returns the same articles with source markers such as "(Reuters)" left in; compare a model trained on each to see how much the markers inflate accuracy. `count_markers()` shows how many articles per class match each marker rule:

In [7]:
from fake_news.data.load import load_clean
from fake_news.data.clean import count_markers

with_markers = load_clean(remove_markers=False)
count_markers(with_markers.sample(5000, random_state=config.SEED))

,real,fake
title_tag,0,439
title_source_suffix,701,32
reuters_dateline,1649,0
reuters_tag,1658,1
agency_credits,37,0
21wire_boilerplate,0,64
scraper_boilerplate,12,36
social_byline,170,27
url,15,473
email,70,7


**Adding a split regime** (for the MinHash pair and Person 2; hand the result to Person 1, who commits `splits.csv`):

```python
from fake_news.data.split import make_group_split, check_split, save_split_column

clean = load_clean()
groups = ...  # pd.Series: doc_id -> near-duplicate group id (or K-means cluster id)
split = make_group_split(clean, groups)  # stratify=False for a few large clusters
check_split(split, labels=clean.set_index("doc_id").is_fake, groups=groups)  # raises if a group leaks
save_split_column("grouped", split)
```

**Paths and constants** live in `fake_news.config` (`SEED`, `SPLIT_FRACTIONS`, `CLEAN_CSV`, `SPLITS_CSV`, `PREDICTIONS_DIR`).